In [ ]:

# NOTE: using this file to train v04. 05/19/2026

import random
from datasets import load_dataset
import torch
import os, json, time, sys
from pathlib import Path
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification,
    AutoTokenizer,
    BertConfig,
    BertForTokenClassification,
    BertTokenizer,
)

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from huggingface_hub import hf_hub_download
# Local imports 
from config import settings 
from metrics import compute_metrics, compute_metrics_complete, plot_metrics
from gcp_utils import upload_to_gcs, verify_upload
from hf_utils import ensure_branch_exists
seed = 18
random.seed(seed)
torch.manual_seed(seed)

def print_title(text,n=50):
    print("="*n)
    print(text)
    print("="*n)

# HF cache directories are now loaded from config.settings
print("HF_HOME:", settings.HF_HOME)
print("HF_DATASETS_CACHE:", settings.HF_DATASETS_CACHE)

# -----------------------------------------------------
# Check Available Device
# -----------------------------------------------------
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")




In [ ]:
dataset = load_dataset(settings.HUGGINGFACE_DATASET_REPO_ID)

In [ ]:
dataset

In [ ]:
dataset = dataset.rename_column("token_label_ids", "labels")
dataset

In [ ]:
train = dataset['train']
validation = dataset['validation']
template_ood = dataset['template_ood']
symptom_ood = dataset['symptom_ood']

In [ ]:
# ============================================================
# TRAIN FUNCTION
# ============================================================
def train(hyperparameters, idx):
    """Train a model with given hyperparameters"""
    MODEL_NAME = hyperparameters["model_name"]
    DATASET_REPO_ID = hyperparameters["dataset_repo"]
    print_title(f"💕 FINETUNING: {MODEL_NAME}             💕")
    print_title(f"💕 Dataset Repo ID: {DATASET_REPO_ID}   💕")


    # -----------------------------------------------------------
    # Load data and id2label/label2id mappings from Hugging Face 
    # -----------------------------------------------------------
    dataset = load_dataset(DATASET_REPO_ID)
    # what data will be loaded from here?